# Leereenheid 4.2: Data-transformasie

## Transformeer data na formate wat geskik is vir analise deur toepaslike tegnieke te gebruik

### Gevallestudie: Boland Meubels & Toestelle

Nadat Pieter sy data skoongemaak het in LU 4.1, moet hy dit nou transformeer sodat dit gereed is vir analise.

In [ ]:
# Laai nodige biblioteke
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler

## Stap 1: Laai die skoon data

Ons begin met die skoon data van LU 4.1.

In [ ]:
# Laai die skoon verkoopsdata
url = 'https://raw.githubusercontent.com/aby-akademia/NGRDA150-2026/main/datastelle/le4_boland_meubels_verkope_skoon.csv'
data = pd.read_csv(url)

# Skakel datum na datetime
data['datum'] = pd.to_datetime(data['datum'])

# Kyk na die eerste paar rye
data.head()

In [ ]:
# Kyk na basiese statistiek
data.describe()

## Stap 2: Normalisering (*Min-Max Scaling*)

Normalisering skaal data na 'n reeks tussen 0 en 1. Dit is nuttig wanneer ons later masjienleer-modelle wil gebruik.

In [ ]:
# Skep 'n kopie van die data
data_getransformeer = data.copy()

# Normaliseer verkoopsprys en kosprys
scaler = MinMaxScaler()

# Pas normalisering toe
data_getransformeer[['verkoopsprys_genormaliseer', 'kosprys_genormaliseer']] = scaler.fit_transform(
    data_getransformeer[['verkoopsprys', 'kosprys']]
)

# Wys die resultaat
print("Oorspronklike vs Genormaliseerde waardes:")
data_getransformeer[['verkoopsprys', 'verkoopsprys_genormaliseer', 'kosprys', 'kosprys_genormaliseer']].head()

## Stap 3: Standaardisering (*Z-score*)

Standaardisering transformeer data sodat dit 'n gemiddelde van 0 en standaardafwyking van 1 het.

In [ ]:
# Standaardiseer hoeveelheid
scaler_std = StandardScaler()

data_getransformeer['hoeveelheid_gestandaardiseer'] = scaler_std.fit_transform(
    data_getransformeer[['hoeveelheid']]
)

# Wys die resultaat
print("Oorspronklike vs Gestandaardiseerde waardes:")
print(data_getransformeer[['hoeveelheid', 'hoeveelheid_gestandaardiseer']].describe())

## Stap 4: Logaritmiese transformasie

Logaritmiese transformasie is nuttig vir data wat regter-skeef is (baie klein waardes, paar groot waardes).

In [ ]:
# Bereken totale inkomste
data_getransformeer['totale_inkomste'] = data_getransformeer['verkoopsprys'] * data_getransformeer['hoeveelheid']

# Pas logaritmiese transformasie toe
# Voeg 1 by om log(0) te vermy
data_getransformeer['log_totale_inkomste'] = np.log(data_getransformeer['totale_inkomste'] + 1)

# Vergelyk verspreiding
print("Oorspronklike totale inkomste:")
print(data_getransformeer['totale_inkomste'].describe())
print("\nLog-getransformeerde totale inkomste:")
print(data_getransformeer['log_totale_inkomste'].describe())

## Stap 5: Een-uit-N kodering (*One-Hot Encoding*)

Een-uit-N kodering skakel kategoriese veranderlikes om na numeriese veranderlikes.

In [ ]:
# Voor een-uit-N kodering
print("Produk:")
print(data_getransformeer['produk'].value_counts())

In [ ]:
# Pas een-uit-N kodering toe op produk_kategorie
een_uit_n = pd.get_dummies(
    data_getransformeer["produk"],
    prefix='tipe',
    drop_first=True
)

# Voeg dit weer by die oorspronklike DataFrame
data_getransformeer = pd.concat([data_getransformeer, een_uit_n], axis=1)

# Wys nuwe kolomme
print("Nuwe kolomme na een-uit-N kodering:")
produk_kolomme = [col for col in data_getransformeer.columns if col.startswith('tipe')]
print(produk_kolomme)

# Wys voorbeelde
data_getransformeer[produk_kolomme].head(10)

## Stap 6: Datum Transformasie

Ons onttrek nuttige komponente uit die datum kolom.

In [ ]:
# Onttrek datum komponente
data_getransformeer['jaar'] = data_getransformeer['datum'].dt.year
data_getransformeer['maand'] = data_getransformeer['datum'].dt.month
data_getransformeer['kwartaal'] = data_getransformeer['datum'].dt.quarter
data_getransformeer['dag_van_week'] = data_getransformeer['datum'].dt.dayofweek

# Wys die resultaat
print("Datum transformasies:")
data_getransformeer[['datum', 'jaar', 'maand', 'kwartaal', 'dag_van_week']].head()

## Stap 7: Finale getransformeerde data

Kom ons kyk na die finale getransformeerde datastel.

In [ ]:
# Kyk na die strukture van die getransformeerde data
print("Getransformeerde datastel:")
print(f"Aantal rye: {len(data_getransformeer)}")
print(f"Aantal kolomme: {len(data_getransformeer.columns)}")
print("\nNuwe kolomme wat geskep is:")
nuwe_kolomme = [
    'verkoopsprys_genormaliseer',
    'kosprys_genormaliseer',
    'hoeveelheid_gestandaardiseer',
    'totale_inkomste',
    'log_totale_inkomste',
    'jaar',
    'maand',
    'kwartaal',
    'dag_van_week'
] + produk_kolomme

for col in nuwe_kolomme:
    print(f"  - {col}")

In [ ]:
# Wys 'n paar voorbeelde
data_getransformeer.head()

## Stap 8: Stoor die getransformeerde data

Stoor die getransformeerde data vir gebruik in LU 4.3.

In [ ]:
# Stoor die getransformeerde data
data_getransformeer.to_csv('boland_meubels_verkope_getransformeer.csv', index=False)
print("Getransformeerde data gestoor as: boland_meubels_verkope_getransformeer.csv")

## Opsomming

In hierdie notaboek het ons:
1. Die skoon data van LU 4.1 gelaai
2. Normalisering toegepas (*Min-Max scaling*) op verkoopsprys en kosprys
3. Standaardisering toegepas (*Z-score*) op hoeveelheid
4. Logaritmiese transformasie toegepas op totale inkomste
5. Een-uit-N kodering toegepas op produk_kategorie
6. Datum komponente onttrek (jaar, maand, kwartaal, dag van week)
7. Die getransformeerde data gestoor vir verdere analise

Pieter se data is nou gereed vir kenmerk-ingenieurswese in LU 4.3.